## Import

In [1]:
import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, average_precision_score)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
TASK = "classification"
N_TRIALS = 30
INNER_K = 3
K, REPEATS = 5, 3
np.random.seed(SEED)
os.makedirs("weights_V7_1", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

cleaned: (2236, 28) | Response rate: 0.1494
features: 17 numeric + 5 dummies = 22 dims


In [4]:
# Feature interaction hand-off (Feature Interaction section of classify.ipynb):
# the strongest interacting features (Teenhome, NumDealsPurchases, NumWebVisitsMonth) are not in the 22-dim model set.
EXTRA = ["Teenhome", "NumDealsPurchases", "NumWebVisitsMonth"]
X_full = pd.concat([X_full, df[EXTRA].astype(float)], axis=1)
ALL_COLS = list(X_full.columns)
BASE_COLS = ALL_COLS[:len(FEATURE_COLS)]
X_all = X_full.values
y_all = y_full.astype(int)

def top_pairs(csv_path, cols, k):
    """Top-k significant (q < 0.05) pairs by effect size where both features are in `cols`."""
    t = pd.read_csv(csv_path)
    t = t[(t.q < 0.05) & t.A.isin(cols) & t.B.isin(cols)].sort_values("effect", ascending=False)
    return list(zip(t.A.head(k), t.B.head(k)))

PAIRS = top_pairs("results/interaction_clf.csv", BASE_COLS + EXTRA, 5)
print("pair features:", PAIRS)
SETS = {
    "base (22)":                  dict(cols=BASE_COLS, pairs=[]),
    "+ 3 modifiers (25)":         dict(cols=BASE_COLS + EXTRA, pairs=[]),
    "+ 3 modifiers + 5 pairs":    dict(cols=BASE_COLS + EXTRA, pairs=PAIRS),
}
inc_name = "Income"

def prep_split(tr_idx, te_idx, set_name):
    """Median-fill Income, standardise, then append standardised products for the pair features (training rows only)."""
    cols, pairs = SETS[set_name]["cols"], SETS[set_name]["pairs"]
    ci = [ALL_COLS.index(c) for c in cols]
    A, B = X_all[tr_idx][:, ci].copy(), X_all[te_idx][:, ci].copy()
    j = cols.index(inc_name); med = np.nanmedian(A[:, j])
    for a in (A, B):
        a[:, j] = np.where(np.isnan(a[:, j]), med, a[:, j])
    sc = StandardScaler().fit(A); A, B = sc.transform(A), sc.transform(B)
    if pairs:
        ix = {c: i for i, c in enumerate(cols)}
        PA = np.column_stack([A[:, ix[a]] * A[:, ix[b]] for a, b in pairs])
        PB = np.column_stack([B[:, ix[a]] * B[:, ix[b]] for a, b in pairs])
        sp = StandardScaler().fit(PA); A, B = np.hstack([A, sp.transform(PA)]), np.hstack([B, sp.transform(PB)])
    return A, B, y_all[tr_idx], y_all[te_idx]

idx_all = np.arange(len(y_all))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED, stratify=y_all)
for s_ in SETS:
    a_, b_, _, _ = prep_split(tr_idx, te_idx, s_)
    print(f"{s_:26s} train {a_.shape} test {b_.shape}")

pair features: [('Teenhome', 'NumWebVisitsMonth'), ('NumDealsPurchases', 'NumStorePurchases'), ('Income', 'NumDealsPurchases'), ('NumDealsPurchases', 'NumWebVisitsMonth'), ('MntSweetProducts', 'NumStorePurchases')]
base (22)                  train (1788, 22) test (448, 22)
+ 3 modifiers (25)         train (1788, 25) test (448, 25)
+ 3 modifiers + 5 pairs    train (1788, 30) test (448, 30)


## Model

In [5]:
POS_W = float((y_all == 0).sum() / (y_all == 1).sum())
# Interaction-depth ladder: the same LightGBM, only allowed to combine 1, 2 or many features per tree.
#   stumps  (1 split per tree)  -> purely additive model, the same family as NAM
#   depth 2 (2 splits per path) -> additive + pairwise interactions, the same family as NA2M
#   full    (V5 run, many leaves) -> any interaction order
import lightgbm as lgb
LGB_CLS = lgb.LGBMClassifier if TASK == "classification" else lgb.LGBMRegressor
def lgb_spec(depth, tier):
    def space(t):
        p = dict(n=t.suggest_int("n", 100, 1000, step=100), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                 mcs=t.suggest_int("mcs", 5, 50), sub=t.suggest_float("sub", 0.6, 1.0), col=t.suggest_float("col", 0.5, 1.0),
                 lam=t.suggest_float("lam", 1e-2, 10, log=True))
        if TASK == "classification":
            p["spw"] = t.suggest_categorical("spw", [1.0, POS_W])
        return p
    def make(p, s):
        extra = dict(scale_pos_weight=p["spw"]) if TASK == "classification" else {}
        return LGB_CLS(n_estimators=p["n"], learning_rate=p["lr"], max_depth=depth, num_leaves=2 ** depth,
                       min_child_samples=p["mcs"], subsample=p["sub"], subsample_freq=1, colsample_bytree=p["col"],
                       reg_lambda=p["lam"], n_jobs=1, random_state=s, verbosity=-1, **extra)
    d = dict(tier=tier, space=space, make=make)
    if TASK == "classification":
        d["kind"] = "proba"
    return d
TIER = "curriculum" if TASK == "classification" else "curriculum family"
LADDER = {"LightGBM stumps (additive, depth 1)": lgb_spec(1, TIER),
          "LightGBM depth 2 (pairwise)": lgb_spec(2, TIER)}

In [6]:
def score_of(est, X, kind):
    return est.predict_proba(X)[:, 1] if kind == "proba" else est.decision_function(X)

def inner_oof(spec, params, X, y, seed):
    """Out-of-fold scores on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in StratifiedKFold(INNER_K, shuffle=True, random_state=seed).split(X, y):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = score_of(est, X[b], spec["kind"])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) maximising average precision of the inner out-of-fold scores."""
    def objective(trial):
        return average_precision_score(y, inner_oof(spec, spec["space"](trial), X, y, seed))
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=spec.get("trials", N_TRIALS))
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def best_threshold(y, s):
    """F1-maximising threshold, searched on training-side out-of-fold scores only."""
    cand = np.unique(np.quantile(s, np.linspace(0.40, 0.99, 120)))
    f = [f1_score(y, (s >= t).astype(int), zero_division=0) for t in cand]
    return float(cand[int(np.argmax(f))])

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    """Threshold from inner OOF on (Xa, ya); refit on all of (Xa, ya); score (Xb, yb)."""
    thr = best_threshold(ya, inner_oof(spec, params, Xa, ya, seed))
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    s = score_of(est, Xb, spec["kind"])
    pred = (s >= thr).astype(int)
    row = {"f1": f1_score(yb, pred, zero_division=0), "accuracy": accuracy_score(yb, pred),
           "precision": precision_score(yb, pred, zero_division=0),
           "recall": recall_score(yb, pred, zero_division=0),
           "roc_auc": roc_auc_score(yb, s), "threshold": thr}
    if spec["kind"] == "proba":
        row["bce"] = log_loss(yb, np.clip(s, 1e-12, 1 - 1e-12))
    return row, s

## Sanity Check

In [7]:
a, b = next(StratifiedKFold(K, shuffle=True, random_state=SEED).split(X_all, y_all))
Xa, Xb, ya, yb = prep_split(a, b, "+ 3 modifiers (25)")
for name, spec in LADDER.items():
    est = spec["make"](dict(n=50, lr=0.1, mcs=20, sub=1.0, col=1.0, lam=1.0, spw=1.0), SEED).fit(Xa, ya)
    leaves = est.booster_.dump_model()["tree_info"][0]["num_leaves"]
    print(f"{name:38s} leaves per tree <= {leaves}")
    assert leaves <= 2 ** est.get_params()["max_depth"]
print("feature set:", "+ 3 modifiers (25)", Xa.shape)

LightGBM stumps (additive, depth 1)    leaves per tree <= 2
LightGBM depth 2 (pairwise)            leaves per tree <= 4
feature set: + 3 modifiers (25) (1788, 25)


## Training

In [8]:
cache = "weights_V7_1/_progress_classification_lightgbm.pkl"
state = pickle.load(open(cache, "rb")) if os.path.exists(cache) else {"rows": [], "scores": {}}
done = {(r["model"], r["repeat"], r["fold"]) for r in state["rows"]}
for rep in range(1, REPEATS + 1):
    seed_r = SEED + rep - 1               # same outer folds as V4_1 / V5_1 / V6_1
    for fold, (a, b) in enumerate(StratifiedKFold(K, shuffle=True, random_state=seed_r).split(X_all, y_all), start=1):
        Xa, Xb, ya, yb = prep_split(a, b, "+ 3 modifiers (25)")
        for base, spec in LADDER.items():
            name = f"{base} | + 3 modifiers (25)"
            if (name, rep, fold) in done:
                continue
            t0 = time.time()
            params = tune(spec, Xa, ya, seed_r * 100 + fold)
            row, s = fit_eval(spec, params, Xa, ya, Xb, yb, seed_r * 100 + fold)
            state["rows"].append(dict(model=name, tier=spec["tier"], selection="inner validation", repeat=rep, fold=fold,
                                      n_test=int(len(yb)), **{k: round(float(v), 4) for k, v in row.items()},
                                      params=params, seconds=round(time.time() - t0, 1)))
            state["scores"].setdefault(f"{name}|r{rep}", np.full(len(y_all), np.nan))[b] = s
        pickle.dump(state, open(cache, "wb"))
        print(f"repeat {rep} fold {fold} done", flush=True)

repeat 1 fold 1 done


repeat 1 fold 2 done


repeat 1 fold 3 done


repeat 1 fold 4 done


repeat 1 fold 5 done


...

repeat 3 fold 1 done


repeat 3 fold 2 done


repeat 3 fold 3 done


repeat 3 fold 4 done


repeat 3 fold 5 done


## K-Fold Summary

In [9]:
kdf = pd.DataFrame(state["rows"])
KEYS = ['f1', 'accuracy', 'precision', 'recall', 'roc_auc', 'bce']
summary = {m: {k: {"mean": round(float(g[k].mean()), 4), "std": round(float(g[k].std()), 4)} for k in KEYS if k in g and g[k].notna().any()}
           for m, g in kdf.groupby("model")}
print(kdf.groupby("model")["f1"].agg(["mean", "std"]).round(4).to_string())
json.dump({"k": K, "repeats": REPEATS, "folds": state["rows"], "summary": summary},
          open("weights_V7_1/classification_lightgbm_kfold_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V7_1/classification_lightgbm_kfold_scores.npz", **state["scores"])

                                                            mean     std
model                                                                   
LightGBM depth 2 (pairwise) | + 3 modifiers (25)          0.5883  0.0545
LightGBM stumps (additive, depth 1) | + 3 modifiers (25)  0.5759  0.0367
